## Embedding Text

- query will be embedded using onnx model

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
#load the dataset

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        d1 = doc['content']
        break

In [5]:
dv = model.encode(d1)

In [6]:
#create an embedded query

q1 = "How does approximate nearest neighbor search work?"
v1 = model.encode(q1)

In [9]:
v1[0] #Q1

np.float32(-0.020582044)

In [ ]:
#calculate the similary between query and the docs

dv.dot(v1)

np.float32(0.4568134)

- The cosine similarity between the documents and the query is 0.46 

## Chunking Data

In [10]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
len(chunks)

295

In [ ]:
#embed the chunks

from embedder import Embedder
from tqdm.auto import tqdm
embedder = Embedder()

vectors = []

for chunk in tqdm(chunks):
    batch_vectors = embedder.encode(chunk['content'])
    vectors.append(batch_vectors)

2026-06-27 00:53:55.186705013 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


  0%|          | 0/295 [00:00<?, ?it/s]

## Vector Search using numpy

In [ ]:
#turn it into array
import numpy as np
X = np.array(vectors)

In [ ]:
X.shape 

(295, 384)

In [ ]:
#callculate all the scores of chunks against the question/query
q1 = "How does approximate nearest neighbor search work?"
v1 = embedder.encode(q1)

scores = X.dot(v1)

In [ ]:
#find the highest match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489017718578813))

In [ ]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [ ]:
#select top 5 highest similarity
top5 = np.argsort(-scores)[:5]
top5

array([ 94,  14, 162,  85, 253])

In [ ]:
for idx in top5:
    print(scores[idx])
    print(chunks[idx])
    print()

0.6489017718578813
{'start': 1000, 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data l

## Minsearch for VectorSearch

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)

In [ ]:
query = "What metric do we use to evaluate a search engine?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

## Textsearch using minsearch

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['content']
)

index.fit(documents)

In [ ]:
query = "How do I store vectors in PostgreSQL?"

search_results = index.search(
    query,
    num_results=5
)

In [ ]:
for result in search_results:
    print(result['filename'])

02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/03-embeddings-dataset.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/10-next-steps.md


### Search + Vector Search (RRF)

In [ ]:
v_search = embedder.encode(query)
scores = X.dot(v_search)

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(100), np.float64(0.5524407406405449))

In [ ]:
top5 = np.argsort(-scores)[:5]
top5

array([100, 103, 127, 101, 102])

In [ ]:
for idx in top5:
    print(scores[idx])
    print(chunks[idx]['filename'])
    print()

0.5524407406405449
02-vector-search/lessons/08-pgvector.md

0.5305404499113289
02-vector-search/lessons/08-pgvector.md

0.46856585584239385
03-orchestration/lessons/05-rag.md

0.46439224632795484
02-vector-search/lessons/08-pgvector.md

0.44903585630485676
02-vector-search/lessons/08-pgvector.md



In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query = "How do I give the model access to tools?"

search_results = index.search(
    query,
    num_results=5
)

search_results

[{'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson we built a RAG pipeline with `RAGBase.rag()`\nand saw it fail on the "Olama" typo. The search returned nothing\nuseful, and the LLM had no way to recover.\n\nThe flow that broke:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    S[search - Olama - no useful results]\n    A([LLM: I don\'t have information about Olama.])\n\n    U --> S --> A\n```\n\nThe pipeline is fixed: search, build prompt, LLM.\n\n```python\ndef rag(self, query):\n    search_results = self.search(query)\n    prompt = self.build_prompt(query, search_results)\n    answer = self.llm(prompt)\n    return answer\n```\n\nThe LLM is a passenger here, not a driver. It never sees the bad\nsearch results, so it can\'t try again with a corrected query.\n\n## The agent alternative\n\nAn agent puts the LLM in charge.\n\nInstead of running se

In [ ]:
query_vector = model.encode(query)
vector_results = vindex.search(query_vector, num_results=5)
vector_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [ ]:
results = rrf([vector_results, search_results])
results

[{'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson we built a RAG pipeline with `RAGBase.rag()`\nand saw it fail on the "Olama" typo. The search returned nothing\nuseful, and the LLM had no way to recover.\n\nThe flow that broke:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    S[search - Olama - no useful results]\n    A([LLM: I don\'t have information about Olama.])\n\n    U --> S --> A\n```\n\nThe pipeline is fixed: search, build prompt, LLM.\n\n```python\ndef rag(self, query):\n    search_results = self.search(query)\n    prompt = self.build_prompt(query, search_results)\n    answer = self.llm(prompt)\n    return answer\n```\n\nThe LLM is a passenger here, not a driver. It never sees the bad\nsearch results, so it can\'t try again with a corrected query.\n\n## The agent alternative\n\nAn agent puts the LLM in charge.\n\nInstead of running se